# Corrected PM2.5 revision: evidence overview

This notebook displays saved results from actual training scripts. It does not run model training. All 120 main experiment tasks and eight controls completed; broad-input SARIMAX failed bounded convergence and has no accepted forecasts. The source `main.ipynb`, raw CSV, submitted manuscript and earlier revision evidence are preserved.

Start with [the review](reports/PHASE4A_REVIEW.md), [reviewer ledger](reports/REVIEWER_EVIDENCE.md) and [author decision](AUTHOR_DECISION.md).

## Pipeline

Raw CSV → hourly calendar → mask four invalid pollutant input cells → chronological training-only feature analysis and validation → train-only scaling and model-specific fitting → 23 weekly forecasts in frozen and expanding-refit regimes → common observed-hour scoring → causal EWMA and controls → uncertainty, interpretation, figures and saved-evidence audits.

The actual future gases are supplied under Perfect Prognosis. Fitting/scoring targets are never imputed. Missing inference context is filled causally under the documented policy; four test weeks have partial scoring coverage. [Protocol](PROTOCOL.md) · [Implementation](code/coverage_runner.py) · [Data science flow](../flow.md).

In [1]:
from pathlib import Path
import json
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)
repo = Path.cwd().resolve()
while not (repo / 'revision/corrected/artifacts').exists():
    if repo.parent == repo:
        raise RuntimeError('Open this notebook within the project repository')
    repo = repo.parent
root = repo / 'revision/corrected'
artifacts = root / 'artifacts'
display(json.loads((artifacts / 'phase3/postfit_audit.json').read_text()))

{'source_sha256': '77e164b3b74cc75adb7293803f69ba090b96ac7eaf1e5ddbd6179be8d95e3e64', 'protocol_signature': 'cc1a6f6d34887f37052db3fb351fae65d6414e24c5e0e08f0061bf983382cdc7', 'validation_signature': 'd7337574e73bbd3f69d8ffa161cb9ffab08496e92a7cf73b42e01b8520b3f9fc', 'successful_validation_candidates': 7, 'corrected_runs_checked': 128, 'pre_origin_scalers_checked': 125, 'neural_episode_sample_counts_checked': 75, 'identical_first_origin_reuses_checked': 5, 'performance_streams': 26, 'separately_signed_numerical_recoveries_checked': 0, 'declared_failed_controls_checked': ['ablate_sarimax_broad'], 'all_tasks_successful': False, 'neuralprophet_seed_summary_checked': True, 'component_hour_weighting_checked': True, 'reused_forecast_source_cost_checked': True, 'figure_pairs_checked': 6, 'primary_origins': 23, 'scored_hours': 3624, 'ready_for_author_evidence_review': True}

## Input correction and training-only selection

Only four invalid pollutant predictor cells become missing in memory. Target values, negative weather values, the 23 origins and 3,624 scored hours are unchanged. All candidates were retrained on corrected inputs before selection; the selected settings remain the same.

In [2]:
display(pd.read_csv(artifacts / 'phase1/invalid_input_changes.csv'))

timestamp,input,original,corrected,partition
2024-08-05 03:00:00,no2,-9999.0,NaN,training
2024-08-11 00:00:00,no2,-9999.0,NaN,training
2023-07-26 07:00:00,o3,-9999.0,NaN,training
2024-02-07 23:00:00,pm10,-9999.0,NaN,training


In [3]:
selection = json.loads((artifacts / 'phase2/selection.json').read_text())
display(selection['selected'])

{'sarimax': {'order': [1, 0, 1], 'seasonal_order': [1, 0, 1, 24]}, 'prophet': {'seasonality_mode': 'additive'}, 'neuralprophet': {'epochs': 50}}

## Main forecast errors

Pooled MAE/RMSE weight scored hours equally; mean weekly MAE weights the 23 origins equally. Units are μg/m³. The individual model rows below use seed 42. EWMA rows use predefined α=0.3 and bias known before each forecast origin.

In [4]:
performance = pd.read_csv(artifacts / 'phase3/performance.csv')
names = [f'{f}_{r}_s42' for f in ['sarimax','prophet','neuralprophet'] for r in ['frozen','walk']]
names += [f'{f}_frozen_s42_ewma03' for f in ['sarimax','prophet','neuralprophet']]
names += ['persistence','daily_persistence','weekly_persistence']
display(performance.set_index('stream').loc[names].reset_index()[['stream','hours','pooled_mae','pooled_rmse','mean_weekly_mae']].round(3))

stream,hours,pooled_mae,pooled_rmse,mean_weekly_mae
sarimax_frozen_s42,3624,30.484,46.964,30.658
sarimax_walk_s42,3624,30.430,46.878,30.597
prophet_frozen_s42,3624,43.929,57.560,44.202
prophet_walk_s42,3624,38.720,51.135,39.012
neuralprophet_frozen_s42,3624,53.392,70.169,53.121
neuralprophet_walk_s42,3624,41.949,56.512,42.383
sarimax_frozen_s42_ewma03,3624,29.938,43.802,30.166
prophet_frozen_s42_ewma03,3624,37.687,49.969,38.068
neuralprophet_frozen_s42_ewma03,3624,37.618,50.908,38.290
persistence,3624,125.738,198.327,127.777


## Seed variability and controls

NeuralProphet varies across seeds. EWMA improves seeds 42/123 but slightly worsens seed 2026. Broad controls supply actual future PM10; their errors cannot establish an operational advantage. The SARIMAX broad control is absent because neither bounded fit converged.

In [5]:
display(pd.read_csv(artifacts / 'phase3/neuralprophet_seed_summary.csv').round(3))

regime,seed_count,pooled_mae_mean,pooled_mae_sd,weekly_mae_mean,weekly_mae_sd_across_seeds
frozen,3,46.883,10.398,47.058,9.764
walk,3,39.605,4.325,40.180,4.075


In [6]:
display(performance[performance.stream.str.endswith(('_no_inputs','_broad','_clip'))][['stream','hours','pooled_mae','pooled_rmse']].round(3))

stream,hours,pooled_mae,pooled_rmse
sarimax_frozen_no_inputs,3624,115.238,173.535
sarimax_frozen_clip,3624,33.167,51.546
prophet_frozen_no_inputs,3624,134.067,190.396
prophet_frozen_broad,3624,15.869,27.572
prophet_frozen_clip,3624,47.679,62.761
neuralprophet_frozen_no_inputs,3624,119.499,177.023
neuralprophet_frozen_broad,3624,22.345,33.292
neuralprophet_frozen_clip,3624,58.280,75.951


## Saved figures

![Both regimes](artifacts/phase3/figures/regime_comparison.png)

![All-week errors](artifacts/phase3/figures/weekly_mae_chronology.png)

All six PDF/PNG pairs are in [the figure directory](artifacts/phase3/figures/). The [analysis report](reports/PHASE3_RESULTS.md) links every figure and explains matched comparisons, seed variability, high-event errors, components and timing boundaries. Final manuscript captions are a writing task.

## Decision checkpoint

The corrected saved-evidence audit and 25 contract tests pass, with the declared failed control explicitly excluded. The all-task success flag remains false. API provenance, historical context filling, limited seasonal coverage, correlated predictor interpretation, seed variability and prior test-period exposure constrain the claims.

Moazzam’s decision is pending in [AUTHOR_DECISION.md](AUTHOR_DECISION.md). No manuscript copy has been made. To regenerate saved analyses without model training: `bash revision/corrected/run_analysis.sh` from the repository root.